# A2：ACC 自适应巡航 —— 跟驰控制
# A2: ACC Adaptive Cruise Control — Car Following

## 一、实验原理 | Principle

### 1.1 什么是 ACC？
**ACC（Adaptive Cruise Control，自适应巡航）**是 L1/L2 自动驾驶的核心功能：
- 无前车时：按设定速度巡航（如 120km/h）
- 有前车时：自动跟驰，保持安全距离，前车慢则慢、前车快则快

与 A1 的 AEB（紧急制动）不同，ACC 追求**平稳舒适**——加速度变化要柔和，
不能让乘客感到顿挫。

### 1.2 跟驰控制原理
**期望间距** = 自车速度 × headway（时距）+ 最小间距
- `headway` 通常 0.8~1.5 秒：跟车时想保持几秒的距离
- 控制器用 **PD 算法**同时修正两个误差：
  - 间距误差：实际间距 vs 期望间距
  - 速度误差：自车速度 vs 前车速度

### 1.3 与 A1 的区别
| | A1 (AEB) | A2 (ACC) |
|---|---|---|
| 目标 | 避免碰撞 | 舒适跟驰 |
| 制动 | 越猛越好（-5m/s²） | 越柔越好（±2m/s²内） |
| 触发 | TTC < 阈值 | 持续工作 |
| 评价 | 最小 TTC | 间距误差、舒适性 |

## 二、实验目的 | Objectives
1. 理解 ACC 的双闭环控制逻辑（间距环 + 速度环）
2. 掌握 PD 参数（kp/kd）对跟驰稳定性的影响
3. 学会在安全性和舒适性之间权衡

## 三、学生任务 | Your Tasks
1. **运行**仿真，观察 ACC 跟驰过程的速度曲线
2. **调参**：`kp` 从 0.8 改为 1.5，观察是否振荡；`kd` 从 0.5 改为 1.0，观察响应速度
3. **记录**：不同参数下的最大间距误差
4. **思考题**：为什么 ACC 的 `kp` 不能像 AEB 那样设得很大？（提示：舒适性 vs 响应速度）

## 四、验证内容 | Verification
- [ ] esmini 成功初始化
- [ ] 20 秒跟驰无碰撞
- [ ] 最大间距误差 ≤ 1.5m
- [ ] 速度曲线平稳无大幅振荡

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `SE_InitWithArgs failed` | esmini 初始化失败 | 检查场景文件路径 |
| 间距误差持续增大 | 控制器发散 | 减小 `kp`，增大 `kd` |
| 速度剧烈振荡 | PD 参数不当 | 减小 `kp` 或增大 `kd` |


> ⚠️ **架构诚实声明**：本实验采用混合架构——Target车辆由esmini引擎加载，但实测发现esmini在ctypes模式下目标车的脚本动作不生效（车辆冻结不动，等价于静态障碍物）；Ego车辆由Python运动学模型（Kinematic Bicycle Model）积分推进，通过`SE_ReportObjectPos`上报给esmini。这不是完整的esmini双车闭环，而是"esmini场景加载 + Python Ego闭环"的教学简化。如需全动态双车，请使用esmini二进制headless模式。


> ⚠️ **Architecture Disclosure**: This experiment uses a hybrid architecture — the Target vehicle is loaded by the esmini engine, but testing revealed that scripted target actions do not take effect in ctypes mode (vehicle frozen, equivalent to a static obstacle); the Ego vehicle is integrated by a Python kinematic bicycle model and reported to esmini via `SE_ReportObjectPos`. This is a teaching simplification ("esmini scene loading + Python Ego closed-loop"), not a full esmini two-vehicle closed loop. For fully dynamic two-vehicle simulation, use the esmini binary in headless mode.



In [ ]:
import sys
sys.dont_write_bytecode = True  # 禁用.pyc缓存
# 清理可能存在的stale .pyc缓存（esmini ctypes对缓存敏感）
import shutil, os
for _p in ['/home/jovyan/labs/common/__pycache__']:
    if os.path.exists(_p):
        shutil.rmtree(_p, ignore_errors=True)
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
from common.esmini_env import EsminiEnv
from common.esmini_env import EsminiEnv
print('✓ 导入成功')


## 1. 实现你的 ACC 控制器 | Implement Your ACC Controller
继承 `BaseController`，实现 PD 跟驰逻辑。
**接口**：输入观测（含前车位置/速度），输出 `{'target_a': 加速度, 'steer_angle': 0}`。
**提示**：`kp` 管间距误差，`kd` 管速度误差。`kp` 太大容易振荡，太小跟不上。


In [ ]:
class MyACCController(BaseController):
    """ACC 自适应巡航（学生实现区）"""
    def __init__(self, config=None):
        super().__init__(config)
        self.headway = self.config.get('headway', 1.0)
        self.kp, self.kd = 0.8, 0.5
    def step(self, observation):
        ego, targets = observation['ego'], observation.get('targets', [])
        if not targets:
            return {'target_a': 0.5, 'steer_angle': 0.0}  # 巡航加速
        lead = min(targets, key=lambda t: abs(t['x'] - ego['x']))
        # 间距误差 + 速度误差 PD 控制
        gap_err = (abs(lead['x'] - ego['x']) - 4.5) - ego['v'] * self.headway
        v_err = lead['v'] - ego['v']
        a = self.kp * gap_err + self.kd * v_err
        return {'target_a': max(-3.0, min(2.0, a)), 'steer_angle': 0.0}
print('✓ ACC 控制器定义完成')


## 2. 真 esmini 闭环仿真 | Real esmini Closed-Loop
启动真实 esmini 引擎，Target 车辆按场景文件行驶，
你的 ACC 控制器每 0.05s 读取前车状态、输出加速度指令。


In [ ]:
controller = MyACCController(config={'headway': 1.0})
env = EsminiEnv(scenario='/home/jovyan/labs/ad2026/cutin.xosc',
                dt=0.05)
obs = env.reset(ego_x=10.0, ego_y=0.0, ego_yaw=0.0, ego_v=15.0)
controller.reset()

max_spacing_error, is_collision = 0.0, False
history = []
for step in range(400):  # 20s
    action = controller.step(obs)
    obs, done, info = env.step(action)
    ego = obs['ego']
    for tgt in obs.get('targets', []):
        gap = abs(tgt['x'] - ego['x']) - 4.5
        err = abs(gap - ego['v'] * 1.0)
        max_spacing_error = max(max_spacing_error, err)
        if gap < 0.5:
            is_collision = True
    history.append((obs['sim_time'], ego['v']))
    if done or is_collision:
        break
env.close()
print(f'✓ 完成：{len(history)} 步，最大间距误差={max_spacing_error:.2f}m，碰撞={is_collision}')


## 3. 评测断言 | Evaluation
- `is_collision` 必须为 False（无碰撞）
- `max_spacing_error` ≤ 1.5m（跟驰够稳）


In [ ]:
assert not is_collision, '发生碰撞!'
assert max_spacing_error <= 1.5, f'间距误差 {max_spacing_error:.2f}m 超标!'
print('✅ 评测通过！')


## 4. 可视化 | Visualization
自车速度曲线（真实仿真数据）：观察跟驰过程中速度是否平稳、有无振荡。


In [ ]:
import matplotlib.pyplot as plt
times = [t for t, v in history]
speeds = [v for t, v in history]
plt.figure(figsize=(10, 4))
plt.plot(times, speeds)
plt.xlabel('Time (s)')
plt.ylabel('Ego Speed (m/s)')
plt.title('ACC Following - Ego Speed')
plt.grid(True, alpha=0.3)
plt.show()


## 五、🏆 挑战任务 | Challenges
1. **舒适性约束**：限制加速度变化率（jerk < 2m/s³），观察跟驰是否更平顺。
2. **headway 对比**：0.8 / 1.0 / 1.2 三档，哪个最省油？（提示：跟车越紧，风阻越小，但越危险）
